In [6]:

import os
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
from sklearn.model_selection import train_test_split

In [7]:

# =============================================================================
# SECTION 1: GPU SETUP
# =============================================================================
print("=" * 80)
print("SECTION 1: GPU SETUP")
print("=" * 80)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Devices: {tf.config.list_physical_devices('GPU')}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("✓ GPU memory growth enabled")
else:
    print("⚠ Warning: No GPU detected. Enable GPU in Notebook Settings → Accelerator.")

policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(f"✓ Mixed precision enabled: {policy.name}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


SECTION 1: GPU SETUP
TensorFlow Version: 2.19.0
GPU Devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
✓ GPU memory growth enabled
✓ Mixed precision enabled: mixed_float16


In [8]:

# =============================================================================
# SECTION 2: DATASET PATH AUTO-DISCOVERY
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 2: DATASET PATH AUTO-DISCOVERY")
print("=" * 80)

KAGGLE_INPUT = '/kaggle/input'

# ── Print the full mounted directory tree ──────────────────────────────────
print("\nFull /kaggle/input tree (up to depth 5):")
for root, dirs, files in os.walk(KAGGLE_INPUT):
    dirs.sort()
    level = root.replace(KAGGLE_INPUT, '').count(os.sep)
    if level > 5:
        continue
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or 'input'}/")
    if level < 5:
        sub = '  ' * (level + 1)
        for f in sorted(files)[:10]:
            print(f"{sub}{f}")
        if len(files) > 10:
            print(f"{sub}… ({len(files) - 10} more files)")


# ── Recursively find a file by name ───────────────────────────────────────
def find_files(root, filename):
    matches = []
    for dirpath, _, files in os.walk(root):
        if filename in files:
            matches.append(os.path.join(dirpath, filename))
    return matches


train_csv_candidates = find_files(KAGGLE_INPUT, 'nyu2_train.csv')
test_csv_candidates  = find_files(KAGGLE_INPUT, 'nyu2_test.csv')

print(f"\nnyu2_train.csv found at : {train_csv_candidates}")
print(f"nyu2_test.csv  found at : {test_csv_candidates}")

if not train_csv_candidates or not test_csv_candidates:
    raise FileNotFoundError(
        "\n❌ Could not find nyu2_train.csv / nyu2_test.csv anywhere under "
        f"{KAGGLE_INPUT}.\n\n"
        "To fix this:\n"
        "  1. Open Notebook Settings (right panel)\n"
        "  2. Click 'Add Data'\n"
        "  3. Search for  soumikrakshit/nyu-depth-v2  and attach it\n"
        "  4. Re-run the notebook\n"
    )

TRAIN_CSV = train_csv_candidates[0]
TEST_CSV  = test_csv_candidates[0]
DATA_ROOT = os.path.dirname(TRAIN_CSV)   # images should be relative to here

print(f"\n✓ TRAIN_CSV : {TRAIN_CSV}")
print(f"✓ TEST_CSV  : {TEST_CSV}")
print(f"✓ DATA_ROOT : {DATA_ROOT}")



SECTION 2: DATASET PATH AUTO-DISCOVERY

Full /kaggle/input tree (up to depth 5):
input/
  datasets/
    soumikrakshit/
      nyu-depth-v2/
        nyu_data/
          data/

nyu2_train.csv found at : ['/kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_train.csv']
nyu2_test.csv  found at : ['/kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_test.csv']

✓ TRAIN_CSV : /kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_train.csv
✓ TEST_CSV  : /kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_test.csv
✓ DATA_ROOT : /kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data


In [9]:

# =============================================================================
# SECTION 3: CONFIGURATION
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 3: CONFIGURATION")
print("=" * 80)


class Config:
    DATA_ROOT  = DATA_ROOT
    TRAIN_CSV  = TRAIN_CSV
    TEST_CSV   = TEST_CSV
    OUTPUT_DIR = '/kaggle/working'

    IMG_HEIGHT       = 256
    IMG_WIDTH        = 256
    IMG_CHANNELS     = 3
    DEPTH_CHANNELS   = 1
    BATCH_SIZE       = 8       # reduce to 4 if OOM
    EPOCHS           = 12
    LEARNING_RATE    = 1e-4
    VALIDATION_SPLIT = 0.1
    MAX_DEPTH        = 10.0   # metres — for scaled display only
    AUGMENT          = True

    MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'best_resunet_depth_model.keras')


config = Config()
print(f"Image size   : {config.IMG_HEIGHT}×{config.IMG_WIDTH}")
print(f"Batch size   : {config.BATCH_SIZE}  |  Epochs: {config.EPOCHS}")
print(f"Model output : {config.MODEL_SAVE_PATH}")




SECTION 3: CONFIGURATION
Image size   : 256×256
Batch size   : 8  |  Epochs: 12
Model output : /kaggle/working/best_resunet_depth_model.keras


In [10]:

# =============================================================================
# SECTION 4: DATA LOADING & PREPROCESSING
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 4: DATA LOADING & PREPROCESSING")
print("=" * 80)


def resolve_image_path(raw_path, data_root):
    """
    The CSVs may store paths in several formats, e.g.:
      data/nyu2_train/bedroom_0001/rgb_00001.jpg
      nyu2_train/bedroom_0001/rgb_00001.jpg
    Try multiple strategies to locate the actual file.
    """
    raw_path = raw_path.strip()

    # Strategy 1 – already absolute
    if os.path.isabs(raw_path) and os.path.exists(raw_path):
        return raw_path

    # Strategy 2 – strip known leading prefixes, join to data_root
    for prefix in ('data/', 'nyu_data/data/', './data/', './'):
        if raw_path.startswith(prefix):
            candidate = os.path.join(data_root, raw_path[len(prefix):])
            if os.path.exists(candidate):
                return candidate

    # Strategy 3 – join directly to data_root
    candidate = os.path.join(data_root, raw_path)
    if os.path.exists(candidate):
        return candidate

    # Strategy 4 – join to KAGGLE_INPUT root
    candidate = os.path.join(KAGGLE_INPUT, raw_path)
    if os.path.exists(candidate):
        return candidate

    return None   # will be filtered out


def load_dataset_from_csv(csv_path, data_root):
    df = pd.read_csv(csv_path, header=None, names=["rgb_path", "depth_path"])

    df["rgb_path"]   = df["rgb_path"].apply(lambda p: resolve_image_path(p, data_root))
    df["depth_path"] = df["depth_path"].apply(lambda p: resolve_image_path(p, data_root))

    df = df.dropna(subset=["rgb_path", "depth_path"])
    df = df[
        df["rgb_path"].apply(os.path.exists) &
        df["depth_path"].apply(os.path.exists)
    ].reset_index(drop=True)

    print(f"✓ Loaded {len(df)} valid pairs from {os.path.basename(csv_path)}")

    if len(df) == 0:
        raw = pd.read_csv(csv_path, header=None, names=["rgb_path", "depth_path"]).head(3)
        print("  ⚠ Sample raw paths from CSV (none resolved — check DATA_ROOT):")
        for _, row in raw.iterrows():
            print(f"    rgb  : {row['rgb_path']}")
            print(f"    depth: {row['depth_path']}")

    return df


def load_and_preprocess_image(rgb_path, depth_path, augment=False):
    # ── RGB ──────────────────────────────────────────────────────────────
    rgb_image = cv2.imread(rgb_path)
    if rgb_image is None:
        return (np.zeros((config.IMG_HEIGHT, config.IMG_WIDTH, 3), np.float32),
                np.zeros((config.IMG_HEIGHT, config.IMG_WIDTH, 1), np.float32))

    rgb_image = cv2.cvtColor(rgb_image, cv2.COLOR_BGR2RGB)
    rgb_image = cv2.resize(rgb_image, (config.IMG_WIDTH, config.IMG_HEIGHT))
    rgb_image = rgb_image.astype(np.float32) / 255.0

    # ── Depth ────────────────────────────────────────────────────────────
    depth_map = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    if depth_map is None:
        return (rgb_image,
                np.zeros((config.IMG_HEIGHT, config.IMG_WIDTH, 1), np.float32))

    depth_map = depth_map.astype(np.float32)
    max_val = depth_map.max()
    if max_val > 0:
        depth_map = depth_map / max_val

    depth_map = cv2.resize(depth_map, (config.IMG_WIDTH, config.IMG_HEIGHT))
    depth_map = np.expand_dims(depth_map, axis=-1)

    # ── Augmentation ─────────────────────────────────────────────────────
    if augment and random.random() > 0.5:
        rgb_image = np.fliplr(rgb_image)
        depth_map = np.fliplr(depth_map)
        if random.random() > 0.5:
            rgb_image = np.clip(rgb_image * random.uniform(0.8, 1.2), 0, 1)

    return rgb_image, depth_map


print("Loading training CSV …")
train_df = load_dataset_from_csv(config.TRAIN_CSV, config.DATA_ROOT)

print("Loading test CSV …")
test_df = load_dataset_from_csv(config.TEST_CSV, config.DATA_ROOT)

if len(train_df) == 0:
    raise ValueError(
        "❌ No valid training pairs found after path resolution.\n"
        "Check the diagnostic output above for raw CSV paths vs DATA_ROOT."
    )

train_df, val_df = train_test_split(
    train_df, test_size=config.VALIDATION_SPLIT, random_state=SEED
)

print(f"\n✓ Dataset split:")
print(f"  Training   : {len(train_df)}")
print(f"  Validation : {len(val_df)}")
print(f"  Test       : {len(test_df)}")



SECTION 4: DATA LOADING & PREPROCESSING
Loading training CSV …
✓ Loaded 50688 valid pairs from nyu2_train.csv
Loading test CSV …
✓ Loaded 654 valid pairs from nyu2_test.csv

✓ Dataset split:
  Training   : 45619
  Validation : 5069
  Test       : 654


In [11]:

# =============================================================================
# SECTION 5: DATA PIPELINE (tf.data)
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 5: DATA PIPELINE")
print("=" * 80)


class DepthDataGenerator:
    def __init__(self, dataframe, batch_size, augment=False, shuffle=True):
        self.dataframe  = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.augment    = augment
        self.shuffle    = shuffle

    def _load_image_pair(self, rgb_path, depth_path):
        def _py_load(rgb_p, depth_p):
            rgb_p   = rgb_p.numpy().decode('utf-8')
            depth_p = depth_p.numpy().decode('utf-8')
            rgb, depth = load_and_preprocess_image(rgb_p, depth_p, self.augment)
            return rgb, depth

        rgb_image, depth_map = tf.py_function(
            _py_load, [rgb_path, depth_path], [tf.float32, tf.float32]
        )
        rgb_image.set_shape([config.IMG_HEIGHT, config.IMG_WIDTH, config.IMG_CHANNELS])
        depth_map.set_shape([config.IMG_HEIGHT, config.IMG_WIDTH, config.DEPTH_CHANNELS])
        return rgb_image, depth_map

    def create_dataset(self):
        ds = tf.data.Dataset.from_tensor_slices((
            self.dataframe['rgb_path'].values,
            self.dataframe['depth_path'].values,
        ))
        if self.shuffle:
            ds = ds.shuffle(buffer_size=1000, seed=SEED)
        ds = ds.map(self._load_image_pair, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(self.batch_size)
        ds = ds.prefetch(tf.data.AUTOTUNE)
        return ds


train_dataset = DepthDataGenerator(train_df, config.BATCH_SIZE, augment=config.AUGMENT, shuffle=True).create_dataset()
val_dataset   = DepthDataGenerator(val_df,   config.BATCH_SIZE, augment=False, shuffle=False).create_dataset()
test_dataset  = DepthDataGenerator(test_df,  config.BATCH_SIZE, augment=False, shuffle=False).create_dataset()
print("✓ Train / Val / Test tf.data pipelines created")



SECTION 5: DATA PIPELINE


I0000 00:00:1773532362.020642      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773532362.027042      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


✓ Train / Val / Test tf.data pipelines created


In [12]:

# =============================================================================
# SECTION 6: RES-UNET MODEL
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 6: RES-UNET MODEL ARCHITECTURE")
print("=" * 80)


def residual_block(inputs, filters, kernel_size=3, batch_norm=True):
    shortcut = inputs

    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(inputs)
    if batch_norm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(x)
    if batch_norm:
        x = layers.BatchNormalization()(x)

    if inputs.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same', kernel_initializer='he_normal')(shortcut)
        if batch_norm:
            shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x


def encoder_block_res(inputs, filters):
    x = residual_block(inputs, filters)
    x = residual_block(x, filters)
    p = layers.MaxPooling2D(pool_size=2)(x)
    return x, p


def decoder_block_res(inputs, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(inputs)
    x = layers.concatenate([x, skip])
    x = residual_block(x, filters)
    x = residual_block(x, filters)
    return x


def build_resunet(input_shape=(256, 256, 3), output_channels=1):
    inputs = layers.Input(shape=input_shape, name='input_rgb')

    s1, p1 = encoder_block_res(inputs, 64)
    s2, p2 = encoder_block_res(p1, 128)
    s3, p3 = encoder_block_res(p2, 256)
    s4, p4 = encoder_block_res(p3, 512)

    b = residual_block(p4, 1024)
    b = residual_block(b, 1024)

    d1 = decoder_block_res(b,  s4, 512)
    d2 = decoder_block_res(d1, s3, 256)
    d3 = decoder_block_res(d2, s2, 128)
    d4 = decoder_block_res(d3, s1, 64)

    outputs = layers.Conv2D(
        output_channels, 1,
        activation='sigmoid', padding='same',
        name='output_depth', dtype='float32'
    )(d4)

    return models.Model(inputs=inputs, outputs=outputs, name='ResUNet_DepthEstimation')


model = build_resunet(
    input_shape=(config.IMG_HEIGHT, config.IMG_WIDTH, config.IMG_CHANNELS),
    output_channels=config.DEPTH_CHANNELS,
)
print(f"✓ Res-UNet built  |  Parameters: {model.count_params():,}")
model.summary()



SECTION 6: RES-UNET MODEL ARCHITECTURE
✓ Res-UNet built  |  Parameters: 63,900,417


Model: "ResUNet_DepthEstimation"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_rgb           │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ input_rgb[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 256,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 256, 256,  │        256 │ input_rgb[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 256, 256,  │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 256, 256,  │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 256, 256,  │     36,928 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 256, 256,  │          0 │ add_1[0][0]     

 Total params: 63,900,417 (243.76 MB)

 Trainable params: 63,870,977 (243.65 MB)

 Non-trainable params: 29,440 (115.00 KB)

In [13]:

# =============================================================================
# SECTION 7: LOSS & METRICS
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 7: LOSS FUNCTION & METRICS")
print("=" * 80)


def depth_loss_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred))


def depth_loss_rmse(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)))


def depth_accuracy_delta1(y_true, y_pred):
    eps = 1e-6
    thresh = tf.maximum((y_true + eps) / (y_pred + eps),
                        (y_pred + eps) / (y_true + eps))
    return tf.reduce_mean(tf.cast(thresh < 1.25, tf.float32))


def depth_accuracy_delta2(y_true, y_pred):
    eps = 1e-6
    thresh = tf.maximum((y_true + eps) / (y_pred + eps),
                        (y_pred + eps) / (y_true + eps))
    return tf.reduce_mean(tf.cast(thresh < 1.5625, tf.float32))


def depth_accuracy_delta3(y_true, y_pred):
    eps = 1e-6
    thresh = tf.maximum((y_true + eps) / (y_pred + eps),
                        (y_pred + eps) / (y_true + eps))
    return tf.reduce_mean(tf.cast(thresh < 1.953125, tf.float32))


CUSTOM_OBJECTS = {
    'depth_loss_mae':        depth_loss_mae,
    'depth_loss_rmse':       depth_loss_rmse,
    'depth_accuracy_delta1': depth_accuracy_delta1,
    'depth_accuracy_delta2': depth_accuracy_delta2,
    'depth_accuracy_delta3': depth_accuracy_delta3,
}

print("Loss : Mean Absolute Error (MAE)")
print("Acc  : δ<1.25 | δ<1.25² | δ<1.25³")




SECTION 7: LOSS FUNCTION & METRICS
Loss : Mean Absolute Error (MAE)
Acc  : δ<1.25 | δ<1.25² | δ<1.25³


In [14]:

# =============================================================================
# SECTION 8: COMPILATION
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 8: MODEL COMPILATION")
print("=" * 80)

model.compile(
    optimizer=Adam(learning_rate=config.LEARNING_RATE),
    loss=depth_loss_mae,
    metrics=[depth_loss_mae, depth_loss_rmse,
             depth_accuracy_delta1, depth_accuracy_delta2, depth_accuracy_delta3],
)
print(f"✓ Compiled with Adam (lr={config.LEARNING_RATE})")



SECTION 8: MODEL COMPILATION
✓ Compiled with Adam (lr=0.0001)


In [ ]:

# =============================================================================
# SECTION 9: TRAINING
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 9: TRAINING")
print("=" * 80)

callback_list = [
    callbacks.ModelCheckpoint(
        filepath=config.MODEL_SAVE_PATH,
        monitor='val_loss', save_best_only=True, mode='min', verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-7, verbose=1,
    ),
]

print("Starting Res-UNet training …")
start_time = time.time()
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=config.EPOCHS,
    callbacks=callback_list,
    verbose=1,
)
training_time = time.time() - start_time

print(f"\n✓ Training completed in {training_time / 60:.2f} minutes")
print(f"Best val loss  : {min(history.history['val_loss']):.6f}")
print(f"Best val δ<1.25: {max(history.history['val_depth_accuracy_delta1']):.4f}")



SECTION 9: TRAINING
Starting Res-UNet training …
Epoch 1/12


I0000 00:00:1773532394.028244     126 service.cc:152] XLA service 0x7a9e540045b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773532394.028300     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773532394.028308     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773532400.312005     126 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-14 23:53:32.947396: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bias-activation.204 = (f16[8,256,256,64]{3,2,1,0}, u8[0]{0}) custom-call(f16[8,256,256,128]{3,2,1,0} %bitcast.57980, f16[64,1,1,128]{3,2,1,0} %bitcast.50819, f16[64]{0} %bitcast.50822), window={size=1x1}, dim_labels=b01f_o01i->b01f, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="ResUNet_DepthEstimation_1/conv2d_42_1/convolution" source_

5703/5703 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step - depth_accuracy_delta1: 0.4573 - depth_accuracy_delta2: 0.7423 - depth_accuracy_delta3: 0.8789 - depth_loss_mae: 0.1583 - depth_loss_rmse: 0.1953 - loss: 0.1583

2026-03-15 00:33:47.176490: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bias-activation.180 = (f16[5,256,256,64]{3,2,1,0}, u8[0]{0}) custom-call(f16[5,256,256,128]{3,2,1,0} %bitcast.6381, f16[64,1,1,128]{3,2,1,0} %bitcast.6389, f16[64]{0} %bitcast.6392), window={size=1x1}, dim_labels=b01f_o01i->b01f, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="ResUNet_DepthEstimation_1/conv2d_42_1/convolution" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-03-15 00:33:47.580904: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.404522643s
Trying algorithm en


Epoch 1: val_loss improved from inf to 0.11719, saving model to /kaggle/working/best_resunet_depth_model.keras
5703/5703 ━━━━━━━━━━━━━━━━━━━━ 2470s 411ms/step - depth_accuracy_delta1: 0.4573 - depth_accuracy_delta2: 0.7423 - depth_accuracy_delta3: 0.8790 - depth_loss_mae: 0.1583 - depth_loss_rmse: 0.1953 - loss: 0.1583 - val_depth_accuracy_delta1: 0.5619 - val_depth_accuracy_delta2: 0.8252 - val_depth_accuracy_delta3: 0.9353 - val_depth_loss_mae: 0.1172 - val_depth_loss_rmse: 0.1509 - val_loss: 0.1172 - learning_rate: 1.0000e-04
Epoch 2/12
5703/5703 ━━━━━━━━━━━━━━━━━━━━ 0s 364ms/step - depth_accuracy_delta1: 0.6255 - depth_accuracy_delta2: 0.8632 - depth_accuracy_delta3: 0.9503 - depth_loss_mae: 0.1066 - depth_loss_rmse: 0.1385 - loss: 0.1066
Epoch 2: val_loss improved from 0.11719 to 0.08446, saving model to /kaggle/working/best_resunet_depth_model.keras
5703/5703 ━━━━━━━━━━━━━━━━━━━━ 2154s 378ms/step - depth_accuracy_delta1: 0.6255 - depth_accuracy_delta2: 0.8632 - depth_accuracy_de

In [ ]:

# =============================================================================
# SECTION 10: TRAINING VISUALIZATION
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 10: TRAINING VISUALIZATION")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].plot(history.history['loss'],     label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss',   linewidth=2)
axes[0, 0].set(xlabel='Epoch', ylabel='Loss (MAE)', title='Training History — Loss')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history.history['depth_loss_rmse'],     label='Train RMSE', linewidth=2)
axes[0, 1].plot(history.history['val_depth_loss_rmse'], label='Val RMSE',   linewidth=2)
axes[0, 1].set(xlabel='Epoch', ylabel='RMSE', title='Training History — RMSE')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history.history['depth_accuracy_delta1'],     label='Train δ<1.25', linewidth=2)
axes[1, 0].plot(history.history['val_depth_accuracy_delta1'], label='Val δ<1.25',   linewidth=2)
axes[1, 0].set(xlabel='Epoch', ylabel='Accuracy', title='Training History — Accuracy (δ<1.25)')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history.history['val_depth_accuracy_delta1'], label='Val δ<1.25',  linewidth=2, marker='o')
axes[1, 1].plot(history.history['val_depth_accuracy_delta2'], label='Val δ<1.25²', linewidth=2, marker='s')
axes[1, 1].plot(history.history['val_depth_accuracy_delta3'], label='Val δ<1.25³', linewidth=2, marker='^')
axes[1, 1].set(xlabel='Epoch', ylabel='Accuracy', title='Validation Accuracy Metrics')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(config.OUTPUT_DIR, 'resunet_training_history.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {save_path}")


In [ ]:

# =============================================================================
# SECTION 11: EVALUATION ON TEST SET
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 11: MODEL EVALUATION")
print("=" * 80)

best_model = keras.models.load_model(config.MODEL_SAVE_PATH, custom_objects=CUSTOM_OBJECTS)
print("✓ Best checkpoint loaded")


def compute_depth_metrics(y_true, y_pred):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    eps = 1e-6

    mae     = np.mean(np.abs(y_true_f - y_pred_f))
    rmse    = np.sqrt(np.mean((y_true_f - y_pred_f) ** 2))
    abs_rel = np.mean(np.abs(y_true_f - y_pred_f) / (y_true_f + eps))
    thresh  = np.maximum(y_true_f / (y_pred_f + eps), y_pred_f / (y_true_f + eps))

    return {
        'MAE':     mae,
        'RMSE':    rmse,
        'AbsRel':  abs_rel,
        'δ<1.25':  np.mean(thresh < 1.25),
        'δ<1.25²': np.mean(thresh < 1.5625),
        'δ<1.25³': np.mean(thresh < 1.953125),
    }


y_true_list, y_pred_list = [], []
print("Generating predictions on test set …")
for rgb_batch, depth_batch in tqdm(test_dataset):
    pred_batch = best_model.predict(rgb_batch, verbose=0)
    y_true_list.append(depth_batch.numpy())
    y_pred_list.append(pred_batch)

y_true_all   = np.concatenate(y_true_list, axis=0)
y_pred_all   = np.concatenate(y_pred_list, axis=0)
test_metrics = compute_depth_metrics(y_true_all, y_pred_all)

print("\n" + "=" * 80)
print("RES-UNET — TEST SET RESULTS")
print("=" * 80)
print(f"\nError Metrics (Lower is Better):")
print(f"  MAE           : {test_metrics['MAE']:.6f}")
print(f"  RMSE          : {test_metrics['RMSE']:.6f}")
print(f"  Abs Rel Error : {test_metrics['AbsRel']:.6f}")
print(f"\nScaled Metrics (×{config.MAX_DEPTH} m):")
print(f"  Scaled MAE    : {test_metrics['MAE']  * config.MAX_DEPTH:.4f} m")
print(f"  Scaled RMSE   : {test_metrics['RMSE'] * config.MAX_DEPTH:.4f} m")
print(f"\nAccuracy Metrics (Higher is Better):")
print(f"  δ < 1.25      : {test_metrics['δ<1.25']:.4f}  ({test_metrics['δ<1.25']*100:.2f}%)")
print(f"  δ < 1.25²     : {test_metrics['δ<1.25²']:.4f}  ({test_metrics['δ<1.25²']*100:.2f}%)")
print(f"  δ < 1.25³     : {test_metrics['δ<1.25³']:.4f}  ({test_metrics['δ<1.25³']*100:.2f}%)")
print("=" * 80)



In [ ]:

# =============================================================================
# SECTION 12: PREDICTION VISUALIZATION
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 12: PREDICTION VISUALIZATION")
print("=" * 80)


def visualize_predictions(model, dataset, num_samples=5):
    for rgb_batch, depth_batch in dataset.take(1):
        predictions = model.predict(rgb_batch, verbose=0)
        n = min(num_samples, rgb_batch.shape[0])
        fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
        if n == 1:
            axes = axes.reshape(1, -1)

        for i in range(n):
            axes[i, 0].imshow(rgb_batch[i].numpy())
            axes[i, 0].set_title('Input RGB', fontweight='bold')
            axes[i, 0].axis('off')

            gt  = depth_batch[i].numpy().squeeze()
            im1 = axes[i, 1].imshow(gt, cmap='plasma', vmin=0, vmax=1)
            axes[i, 1].set_title('Ground Truth', fontweight='bold')
            axes[i, 1].axis('off')
            plt.colorbar(im1, ax=axes[i, 1], fraction=0.046)

            pred   = predictions[i].squeeze()
            eps    = 1e-6
            thresh = np.maximum((gt + eps) / (pred + eps), (pred + eps) / (gt + eps))
            acc    = np.mean(thresh < 1.25)
            im2    = axes[i, 2].imshow(pred, cmap='plasma', vmin=0, vmax=1)
            axes[i, 2].set_title(f'Predicted  (δ<1.25: {acc:.3f})', fontweight='bold')
            axes[i, 2].axis('off')
            plt.colorbar(im2, ax=axes[i, 2], fraction=0.046)

        plt.tight_layout()
        sp = os.path.join(config.OUTPUT_DIR, 'resunet_predictions.png')
        plt.savefig(sp, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✓ Saved: {sp}")
        break


visualize_predictions(best_model, test_dataset, num_samples=5)



In [ ]:

# =============================================================================
# SECTION 13: SINGLE-IMAGE INFERENCE
# =============================================================================
print("\n" + "=" * 80)
print("SECTION 13: INFERENCE FUNCTION")
print("=" * 80)


def predict_depth(image_path, model=best_model):
    """Predict a depth map for a single RGB image."""
    rgb = cv2.imread(image_path)
    if rgb is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    rgb  = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
    orig = rgb.copy()
    inp  = cv2.resize(rgb, (config.IMG_WIDTH, config.IMG_HEIGHT)).astype(np.float32) / 255.0
    inp  = np.expand_dims(inp, axis=0)
    return orig, model.predict(inp, verbose=0).squeeze()


sample_path = test_df.iloc[0]['rgb_path']
orig_img, depth_pred = predict_depth(sample_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(orig_img)
axes[0].set_title('Input Image', fontweight='bold', fontsize=14)
axes[0].axis('off')

im = axes[1].imshow(depth_pred, cmap='plasma')
axes[1].set_title('Predicted Depth (Res-UNet)', fontweight='bold', fontsize=14)
axes[1].axis('off')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
sp = os.path.join(config.OUTPUT_DIR, 'resunet_inference_example.png')
plt.savefig(sp, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {sp}")

# =============================================================================
# DONE
# =============================================================================
print("\n" + "=" * 80)
print("RES-UNET PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Architecture : Residual U-Net with skip connections")
print("\nOutput files in /kaggle/working/:")
print("  • best_resunet_depth_model.keras")
print("  • resunet_training_history.png")
print("  • resunet_predictions.png")
print("  • resunet_inference_example.png")
print("=" * 80)
